# Transfer Learning

## Learning Objectives
1. Understand feature extraction: freeze backbone, train only classification head
2. Fine-tune a pretrained ResNet18 on a synthetic 5-class dataset in PyTorch
3. Apply layer-wise learning rate decay for controlled fine-tuning of BERT
4. Quantify transfer learning data efficiency vs training from scratch


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import copy, time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1 — Feature Extraction: Freeze Backbone

In [ ]:
# ── Level 1: simulated feature extraction ────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# Simulate a pretrained backbone: fixed random weights
class PretrainedBackbone(nn.Module):
    """Frozen backbone producing 128-dim feature vectors."""
    def __init__(self, in_dim=64, out_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 256)
        self.fc2 = nn.Linear(256, out_dim)
    def forward(self, x):
        return F.relu(self.fc2(F.relu(self.fc1(x))))

class ClassificationHead(nn.Module):
    def __init__(self, feat_dim=128, n_class=5):
        super().__init__()
        self.fc = nn.Linear(feat_dim, n_class)
    def forward(self, x): return self.fc(x)

backbone = PretrainedBackbone()
head     = ClassificationHead()

# Freeze backbone
for p in backbone.parameters():
    p.requires_grad = False

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in backbone.parameters() if not p.requires_grad)
head_params = sum(p.numel() for p in head.parameters())

print(f"Backbone params (frozen):     {frozen:,}")
print(f"Head params (trainable):      {head_params:,}")
print(f"Trainable ratio:              {head_params/(frozen+head_params):.1%}")
print("\nOnly the lightweight head is updated — fast, avoids overfitting on small data.")

# Quick forward pass
x = torch.randn(16, 64)
with torch.no_grad():
    features = backbone(x)    # frozen backbone: no grad computed
logits   = head(features)     # head: grads computed
print(f"\nInput: {x.shape}  →  Features: {features.shape}  →  Logits: {logits.shape}")


## Level 2 — Fine-tuning ResNet18 Backbone in PyTorch

In [ ]:
# ── Level 2: fine-tune pretrained CNN on 5-class synthetic task ───────────
import torch, copy, time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision.models as models

torch.manual_seed(42)

# Synthetic image-like data (3-channel 32×32)
N_TRAIN, N_VAL, N_CLASS = 400, 100, 5
X_tr = torch.randn(N_TRAIN, 3, 32, 32)
y_tr = torch.randint(0, N_CLASS, (N_TRAIN,))
X_v  = torch.randn(N_VAL,   3, 32, 32)
y_v  = torch.randint(0, N_CLASS, (N_VAL,))
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)

def make_resnet_feature_extractor(n_class=N_CLASS):
    """Load ResNet18, freeze backbone, replace FC head."""
    try:
        model = models.resnet18(weights=None)    # random init (no download)
    except TypeError:
        model = models.resnet18(pretrained=False)
    # Freeze all backbone layers
    for p in model.parameters():
        p.requires_grad = False
    # Replace FC head with task-specific head (trainable)
    in_feats = model.fc.in_features
    model.fc = nn.Linear(in_feats, n_class)     # only this is trainable
    return model

model = make_resnet_feature_extractor().to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}  ({trainable/total:.1%})")

opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

for epoch in range(10):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        try:
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            opt.step()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("OOM: reduce batch_size or image resolution"); break
            raise

model.eval()
with torch.no_grad():
    acc = (model(X_v.to(device)).argmax(1) == y_v.to(device)).float().mean().item()
print(f"Val accuracy after 10 epochs (frozen backbone): {acc:.3f}")
print("(Random init backbone — real pretrained weights would give much higher acc)")


## Real-World Example 1 — BERT Fine-tuning with Frozen Lower Layers

In [ ]:
# ── RW1: fine-tune upper layers only (BERT-style) ────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

# Simulate a 6-layer transformer encoder (BERT-small substitute)
HIDDEN, N_LAYERS, N_CLASS = 256, 6, 2
FREEZE_LAYERS = 4     # freeze bottom 4 of 6 layers, train top 2

class TransformerBlock(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn = nn.MultiheadAttention(hidden, num_heads=4, batch_first=True)
        self.ff   = nn.Sequential(nn.Linear(hidden, hidden * 4), nn.GELU(),
                                  nn.Linear(hidden * 4, hidden))
        self.ln1  = nn.LayerNorm(hidden)
        self.ln2  = nn.LayerNorm(hidden)
    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x    = self.ln1(x + a)
        return self.ln2(x + self.ff(x))

class BERTLike(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb    = nn.Linear(32, HIDDEN)    # simplified: linear proj instead of embed
        self.layers = nn.ModuleList([TransformerBlock(HIDDEN) for _ in range(N_LAYERS)])
        self.cls_fc = nn.Linear(HIDDEN, N_CLASS)

    def forward(self, x):
        h = self.emb(x)
        for layer in self.layers:
            h = layer(h)
        return self.cls_fc(h[:, 0])   # CLS token

model = BERTLike().to(device)

# Freeze bottom layers
for i, layer in enumerate(model.layers):
    if i < FREEZE_LAYERS:
        for p in layer.parameters():
            p.requires_grad = False

frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Frozen:    {frozen:,}  Trainable: {trainable:,}  "
      f"({trainable/(frozen+trainable):.1%} trainable)")

# Train
X = torch.randn(600, 20, 32)   # (batch, seq_len=20, input_features=32)
y = torch.randint(0, N_CLASS, (600,))
loader = DataLoader(TensorDataset(X[:500], y[:500]), batch_size=32, shuffle=True)
X_v, y_v = X[500:].to(device), y[500:].to(device)

opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                         lr=2e-4, weight_decay=1e-2)
for _ in range(15):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        F.cross_entropy(model(xb), yb).backward()
        opt.step()
model.eval()
with torch.no_grad():
    acc = (model(X_v).argmax(1) == y_v).float().mean().item()
print(f"Val accuracy (top-{N_LAYERS-FREEZE_LAYERS} layers + head trainable): {acc:.3f}")


## Real-World Example 2 — Layer-Wise Learning Rate Decay

In [ ]:
# ── RW2: LLRD — lower LR for earlier layers, higher for later layers ──────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

# Re-use BERTLike from RW1 concept
HIDDEN, N_LAYERS, N_CLASS = 128, 4, 2

class Block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc  = nn.Sequential(nn.Linear(d, d * 2), nn.GELU(), nn.Linear(d * 2, d))
        self.ln  = nn.LayerNorm(d)
    def forward(self, x):
        return self.ln(x + self.fc(x))

class SmallBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb    = nn.Linear(16, HIDDEN)
        self.layers = nn.ModuleList([Block(HIDDEN) for _ in range(N_LAYERS)])
        self.fc     = nn.Linear(HIDDEN, N_CLASS)
    def forward(self, x): return self.fc(self.layers[-1](self.layers[-2](self.layers[-3](self.layers[0](self.emb(x))))))

def make_llrd_optimizer(model, base_lr=2e-4, decay=0.5):
    """Layer-wise learning rate decay: LR multiplier = decay^(N_LAYERS - layer_idx)."""
    param_groups = []
    # Embedding: lowest LR
    param_groups.append({"params": model.emb.parameters(),
                         "lr": base_lr * (decay ** N_LAYERS)})
    # Transformer layers: increasing LR toward top
    for i, layer in enumerate(model.layers):
        lr = base_lr * (decay ** (N_LAYERS - 1 - i))
        param_groups.append({"params": layer.parameters(), "lr": lr})
    # Head: highest LR
    param_groups.append({"params": model.fc.parameters(), "lr": base_lr})
    return torch.optim.AdamW(param_groups, weight_decay=1e-2)

# Compare LLRD vs uniform LR
X  = torch.randn(600, 16)
y  = (X[:, 0] + X[:, 2] > 0).long()
ld = DataLoader(TensorDataset(X[:500], y[:500]), batch_size=32, shuffle=True)
Xv, yv = X[500:].to(device), y[500:].to(device)

def train_and_eval(opt_fn_name, epochs=20):
    import copy
    m = SmallBERT().to(device)
    if opt_fn_name == "llrd":
        opt = make_llrd_optimizer(m)
        print(f"  LLRD LRs: emb={opt.param_groups[0]['lr']:.2e}  "
              f"layer0={opt.param_groups[1]['lr']:.2e}  "
              f"head={opt.param_groups[-1]['lr']:.2e}")
    else:
        opt = torch.optim.AdamW(m.parameters(), lr=2e-4)
    for _ in range(epochs):
        m.train()
        for xb, yb in ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            F.cross_entropy(m(xb), yb).backward()
            opt.step()
    m.eval()
    with torch.no_grad():
        return (m(Xv).argmax(1) == yv).float().mean().item()

acc_uniform = train_and_eval("uniform")
acc_llrd    = train_and_eval("llrd")
print(f"\nUniform LR:          val_acc = {acc_uniform:.3f}")
print(f"Layer-wise LR decay: val_acc = {acc_llrd:.3f}")
print("LLRD prevents lower layers from drifting too far from pretrained representations.")


## Real-World Example 3 — Transfer Learning Data Efficiency

In [ ]:
# ── RW3: accuracy vs #labeled examples — transfer vs scratch ─────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import copy

torch.manual_seed(42)

# Simulate pretrained backbone (already converged on large dataset)
class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(32, 128), nn.ReLU(),
                                 nn.Linear(128, 64), nn.ReLU())
    def forward(self, x): return self.enc(x)

# Pre-"train" backbone (simulate representation learning)
backbone_pretrained = Backbone()
with torch.no_grad():
    # Simulate pretrained weights with meaningful structure
    for p in backbone_pretrained.parameters():
        p.normal_(0, 0.5)

class TransferModel(nn.Module):
    def __init__(self, pretrained_backbone, freeze=True):
        super().__init__()
        self.backbone = copy.deepcopy(pretrained_backbone)
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.head = nn.Linear(64, 2)
    def forward(self, x): return self.head(self.backbone(x))

class ScratchModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(32, 128), nn.ReLU(),
                                 nn.Linear(128, 64), nn.ReLU(),
                                 nn.Linear(64, 2))
    def forward(self, x): return self.net(x)

# Full dataset for evaluation
X_all = torch.randn(1500, 32)
y_all = (X_all[:, 0] - X_all[:, 2] > 0).long()
X_val, y_val = X_all[1000:].to(device), y_all[1000:].to(device)

def train_eval(model, X_tr, y_tr, epochs=30):
    ld = DataLoader(TensorDataset(X_tr, y_tr), batch_size=min(32, len(X_tr)), shuffle=True)
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    opt = torch.optim.Adam(trainable_params, lr=1e-3)
    for _ in range(epochs):
        model.train()
        for xb, yb in ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            F.cross_entropy(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return (model(X_val).argmax(1) == y_val).float().mean().item()

n_samples_list = [20, 50, 100, 200, 500]
print(f"{'N labeled':>10}  {'Transfer':>10}  {'Scratch':>10}")
print("-" * 35)
for n in n_samples_list:
    X_tr = X_all[:n].to(device)
    y_tr = y_all[:n].to(device)
    acc_transfer = train_eval(TransferModel(backbone_pretrained).to(device), X_tr, y_tr)
    acc_scratch  = train_eval(ScratchModel().to(device),                     X_tr, y_tr)
    print(f"{n:>10}  {acc_transfer:>10.3f}  {acc_scratch:>10.3f}")
print("\nTransfer learning provides larger gains with very few labeled examples.")
